# Literate Child Indicator (INEP): how the raw layer is obtained

**Publisher:** Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)  
**Official source:** <https://www.gov.br/inep/pt-br/areas-de-atuacao/avaliacao-e-exames-educacionais>

Share of children literate at the right age, with the targets agreed up to 2030, by municipality (municipal network) and by state and Brazil (public network).

**Grain and keys:** Municipality by edition (wide format in trusted, long series in semantic); one row per state plus a Brazil row.

**Released as:**

- `ica-analytics`: semantic (2 tables)
- `ica-raw-trusted`: raw, trusted (4 tables)

Every table and column is documented in the [interactive data map](https://lucas.rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

**Not executed end to end.** The download cells below use URLs recorded in the released tables, but the publisher's download host did not respond when this notebook was written, so the outputs are empty. Run it from your own network.

In [ ]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. Where the files are

INEP publishes the literacy results and targets as spreadsheets; the released tables record the file name in `arquivo_origem`.

This source has no public API in the form used here, so the notebook documents the page and the manual step instead of pretending to automate it.

In [ ]:
print("https://www.gov.br/inep/pt-br/areas-de-atuacao/avaliacao-e-exames-educacionais")

## 2. From the spreadsheet to the raw layer

Download the spreadsheet for the edition, read it with `pandas.read_excel` (or `read_csv`), keep the columns exactly as published and store them as strings. That is the raw layer. The `arquivo_origem` or `source_file` column of the released tables records which file each row came from.

In [ ]:
# Example: read a downloaded spreadsheet exactly as published
# raw = pd.read_excel("resultados_e_metas_municipios_2025.xlsx", dtype=str)

## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://lucas.rangeltech.net/datamap/#/subject/ica).

In [ ]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/ica-raw-trusted", path="trusted__inep_ica_ufs.parquet")
pd.read_parquet(path).head()